# PreProcess Stage 4: 
## OME-Zarr Conversion


## Prerequisites
Ensure that the previous stage has has been run and the following two files are available: 


- `intermediate_grouper.json` — the grouped image data
- `pipeline_stats.json` — the timing and summary information

## Select data paths
Define the config.yaml path, the name of the intermediate json and the name of the output stats json. If unsure, do not change the defaults.
If Stage 1 2 and 3 were ran with default settings, these will already be correct

In [ ]:
!cd ..
CONFIG_PATH = "pre_process/config.yaml"
IN_DATA     = "intermediate_filtered.json"
STATS_FILE  = "pipeline_stats.json"

## Load filtered dataset
The subsequent Cell of code will load the data created in the previous stages. Two sets of settings are needed at this stage: one for the converter itself and one to locate the Zarr arrays.

In [ ]:
import json
from pathlib import Path

from pre_process._pre_process_utils.pipeline_utils import load_config, deserialise_buckets
from pre_process.ome_converter import OmeConverterConfig
from pre_process.zarr_writer import ZarrWriterConfig

config = load_config(CONFIG_PATH)
ome_cfg = OmeConverterConfig.from_dict(config)
writer_cfg = ZarrWriterConfig.from_dict(config)

filtered_buckets = deserialise_buckets(json.loads(Path(IN_DATA).read_text()))
pipeline_stats = json.loads(Path(STATS_FILE).read_text())

total = sum(len(v) for v in filtered_buckets.values())
print(f"Loaded {total} images across {len(filtered_buckets)} buckets")

## Write OME-Zarr
The Zarr files are now grouped together, reducing the overall number of files we have as a whole.
The Ome-Zarr also creates a hierarchical version of the dataset,enhancing both the data loading and the data storage impact. 

In [ ]:
import time
from pre_process.ome_converter import OmeZarrConverter

t0 = time.perf_counter()
converter = OmeZarrConverter(ome_cfg, writer_cfg)
ome_manifests = converter.convert_all(filtered_buckets)
dt = time.perf_counter() - t0

print(f"OME-Zarr conversion complete in {dt:.1f}s")
print(f"Files created: {len(ome_manifests)}")
print()
for m in ome_manifests:
    zip_info = f"  (zip: {m.zip_path})" if m.zip_path else ""
    print(f"  {m.bucket}: {m.n_images} images, {m.pyramid_levels} levels → {m.ome_store_path}{zip_info}")

## Export Results
A record (manifest) is now written of every array that was created and updates the pipeline stats

In [ ]:
from pre_process._pre_process_utils.pipeline_utils import safe_write_json

pipeline_stats["timing"]["ome_convert_s"] = round(dt, 2)
pipeline_stats["ome_manifests"] = [m.to_dict() for m in ome_manifests]

manifest_path = Path(ome_cfg.output_dir) / "pipeline_manifest.json"
safe_write_json(pipeline_stats, str(manifest_path))
print(f"Final manifest written to: {manifest_path}")

## (Opt) Full Pipeline time check

Running the subsequent cell shows how long each stage took across the entire pipeline.

In [ ]:
timing = pipeline_stats.get("timing", {})
print("Pipeline Timing Summary")
print("-" * 35)
for key, val in timing.items():
    if key == "t0":
        continue
    label = key.replace("_", " ").replace(" s", "").title()
    print(f"  {label:<22} {val if val is not None else 'skipped':>8}{'s' if val is not None else ''}")